# HiMoFlow v5.5 — Notebook 01: Training

**Pipeline:** trains all four stages of the v5.5 hierarchical molecular generation model on ZINC250K.

## Two-stage architecture

**Stage 1 — scaffold construction** (three sub-stages):
- **A1** (RingLayoutDiffusionV5_5): `condition → (R, F, L, spiro_pos)` — ring-level layout
- **A3** (BranchTopologyModel): `A1 outputs + condition → (B_size, B_pos, B_parent, B_bond)` — branch trees per ring
- **A2** (RingAtomDiffusion): `decoded scaffold + condition → atom_ids` — element identity per atom

**Stage 2 — functionalization:**
- **Terminal** (FragmentStage2): `scaffold + condition → per-atom fragment_id` — decorate scaffolds with -OH, -CH3, =O, halogens, etc.

After this notebook runs to completion, all four `best_model.pt` checkpoints exist on disk and `02_evaluation.ipynb` can run end-to-end generation + V·U·N.

## Wall-clock budget

| Stage | Capacity | L4 epoch time | Epochs | Total |
|---|---|---|---|---|
| Preprocessing (labels.pkl) | — | — | — | ~5 min |
| A1 | 3M | ~3 min | 40 | ~2 hr |
| A3 | 10M | ~6 min | 100 | ~10 hr |
| A2 | 10M | ~1.5 min | 40 | ~1 hr |
| Terminal | 9M | ~2 min | 40 | ~1.5 hr |
| **Total** | | | | **~15.5 hr** |

Doesn't fit in one Colab session (12 hr cap). Run across 2-3 sessions; **auto-resume handles disconnects**.

## Idempotency: `FORCE_FROM_SCRATCH`

Every artifact-generating step (CSV, labels.pkl, smoke ckpts, full ckpts) **defaults to reusing existing files**. Set a flag in cell 2 to force regeneration of just that artifact. **Auto-resume** loads `latest.pt` and continues training automatically — no flag needed for disconnects.

**To regenerate a full-training checkpoint from epoch 0**, set its flag to `True`. Otherwise training resumes from wherever `latest.pt` left off.

## 1. Setup — Drive mount, paths, hyperparameters

In [6]:
!pip install -q rdkit matplotlib

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = '/content/drive/My Drive/machine-learning/generative/MeanFlow/mean-flow-v5.5-ZINC250K'
except ImportError:
    BASE = '.'

import os, sys
print('BASE:', BASE)
assert os.path.isdir(BASE), f'BASE not found: {BASE}'
if BASE not in sys.path:
    sys.path.insert(0, BASE)
os.chdir(BASE)
print(f'cwd: {os.getcwd()}')

# ── Dataset paths ─────────────────────────────────────────────────
CSV_DIR        = '/content/drive/My Drive/machine-learning/Data/RedDB/ZIN250K'
CSV_PATH       = f'{CSV_DIR}/250k_rndm_zinc_drugs_clean_3.csv'
SMILES_COL     = 'smiles'
CONDITION_COLS = ['logP', 'SAS']
NORMALIZE_COLS = ['logP', 'SAS']

AUGMENTED_CSV  = f'{BASE}/dataset_augmented.csv'
LABELS_PKL     = f'{BASE}/dataset_labels_v5_5.pkl'

# ── Per-stage hyperparameters ─────────────────────────────────────
SEED          = 42
BATCH_SIZE    = 256
LR            = 3e-4
VAL_FRACTION  = 0.05

A1_CAPACITY      = '3M'
A1_NUM_EPOCHS    = 40
A1_CFG_DROP_PROB = 0.10

A3_CAPACITY      = '10M'
A3_NUM_EPOCHS    = 100
A3_CFG_DROP_PROB = 0.30

A2_CAPACITY      = '10M'
A2_NUM_EPOCHS    = 40
A2_CFG_DROP_PROB = 0.10

TERM_CAPACITY    = '9M'
TERM_NUM_EPOCHS  = 40
TERM_NUM_FRAGMENTS = 22       # v5.5-ZINC vocab
TERM_USE_CLASS_WEIGHTS = True

CKPT_SUFFIX    = ''

# ── Derived checkpoint paths ──────────────────────────────────────
A1_CKPT   = f'{BASE}/checkpoints_a1_v5_5_{A1_CAPACITY}{CKPT_SUFFIX}'
A3_CKPT   = f'{BASE}/checkpoints_a3_v5_5_{A3_CAPACITY}{CKPT_SUFFIX}'
A2_CKPT   = f'{BASE}/checkpoints_a2_v5_5_{A2_CAPACITY}{CKPT_SUFFIX}'
TERM_CKPT = f'{BASE}/checkpoints_terminal_v5_5_{TERM_CAPACITY}{CKPT_SUFFIX}'

A1_SMOKE_CKPT   = A1_CKPT   + '_smoke'
A3_SMOKE_CKPT   = A3_CKPT   + '_smoke'
A2_SMOKE_CKPT   = A2_CKPT   + '_smoke'
TERM_SMOKE_CKPT = TERM_CKPT + '_smoke'

for p in (A1_CKPT, A3_CKPT, A2_CKPT, TERM_CKPT):
    os.makedirs(p, exist_ok=True)

# ── FORCE_FROM_SCRATCH flags ──────────────────────────────────────
# Default: every step REUSES existing artifacts. Set a flag to True
# to force regeneration of just that artifact.
#
# Auto-resume handles disconnects — no flag needed. Flags below only
# matter when you want to START OVER from epoch 0.
FORCE_FROM_SCRATCH = {
    # Preprocessing
    'augmented_csv':  False,
    'labels_pkl':     False,
    # Stage 1A — A1
    'a1_smoke_ckpt':  False,
    'a1_ckpt':        True,    # ⚠️ deletes A1 progress
    # Stage 1B — A3
    'a3_smoke_ckpt':  False,
    'a3_ckpt':        True,    # ⚠️ deletes A3 progress
    # Stage 1C — A2
    'a2_smoke_ckpt':  False,
    'a2_ckpt':        True,    # ⚠️ deletes A2 progress
    # Stage 2 — Terminal
    'term_smoke_ckpt': False,
    'term_ckpt':       True,   # ⚠️ deletes Terminal progress
}

print(f'\nDataset paths:')
print(f'  CSV:        {CSV_PATH}')
print(f'  augmented:  {AUGMENTED_CSV}')
print(f'  labels:     {LABELS_PKL}')
print(f'\nCheckpoint paths:')
for n, p in [('A1', A1_CKPT), ('A3', A3_CKPT), ('A2', A2_CKPT), ('Terminal', TERM_CKPT)]:
    print(f'  {n:8s} {p}')

print(f'\nFORCE_FROM_SCRATCH (default = reuse all):')
for k, v in FORCE_FROM_SCRATCH.items():
    marker = '⚠️ REGEN ' if v else '  reuse  '
    print(f'  {marker} {k}')

## 2. Clear stale module imports

Run this after editing any `.py` file in Drive. Not needed after a kernel restart.

In [8]:
import sys
to_clear = [k for k in list(sys.modules)
            if 'meanflow' in k
            or 'preprocessing' in k
            or k.startswith('run_training')]
for k in to_clear:
    del sys.modules[k]
print(f'Cleared {len(to_clear)} stale module(s)')

Cleared 6 stale module(s)


## 3. Preprocessing — augmented CSV with Z-scored conditioning

Reads ZINC250K CSV, drops rows missing logP/SAS, Z-score normalizes both columns. Output: `dataset_augmented.csv`. Skipped if exists.

In [9]:
import pandas as pd
import numpy as np

if FORCE_FROM_SCRATCH['augmented_csv'] and os.path.exists(AUGMENTED_CSV):
    print(f'FORCE_FROM_SCRATCH["augmented_csv"]=True → deleting {AUGMENTED_CSV}')
    os.remove(AUGMENTED_CSV)

if os.path.exists(AUGMENTED_CSV):
    print(f'✓ Augmented CSV exists at {AUGMENTED_CSV}, reusing.')
    print(f'  size: {os.path.getsize(AUGMENTED_CSV)/1e6:.1f} MB')
else:
    assert os.path.isfile(CSV_PATH), f'CSV not found: {CSV_PATH}'
    df = pd.read_csv(CSV_PATH)
    print(f'  raw rows: {len(df):,}')
    mask = pd.Series(True, index=df.index)
    for col in CONDITION_COLS:
        assert col in df.columns, f'Missing column {col!r}'
        mask &= df[col].notna()
    df = df.loc[mask].copy()
    print(f'  after dropping missing: {len(df):,}')
    for col in NORMALIZE_COLS:
        if col not in df.columns: continue
        mu, sd = float(df[col].mean()), float(df[col].std())
        df[f'{col}_norm'] = ((df[col] - mu) / sd).astype(np.float32)
        print(f'  {col}: mean={mu:.4f} std={sd:.4f}')
    df.to_csv(AUGMENTED_CSV, index=False)
    print(f'\n  wrote {AUGMENTED_CSV}  ({os.path.getsize(AUGMENTED_CSV)/1e6:.1f} MB)')

df_check = pd.read_csv(AUGMENTED_CSV, nrows=1)
cond_cols_resolved = tuple(
    f'{c}_norm' if f'{c}_norm' in df_check.columns else c
    for c in CONDITION_COLS
)
print(f'\nusing cond_cols = {cond_cols_resolved}')

✓ Augmented CSV exists at /content/drive/My Drive/machine-learning/generative/MeanFlow/mean-flow-v5.5-ZINC250K/dataset_augmented.csv, reusing.
  size: 28.0 MB

using cond_cols = ('logP_norm', 'SAS_norm')


## 4. Preprocessing — extract v5.5 labels

Iterates `extract_layout_v5_5()` over the augmented CSV. Expected ZINC250K retention: ~93.4%. Skipped if exists; takes ~5 min on Colab.

The labels pkl is the only persistent dataset artifact — all four stages train on the **same labels.pkl**, just using different fields. ~3.2 GB on disk.

In [10]:
import pandas as pd, numpy as np, pickle, time
from collections import defaultdict
from preprocessing.ring_layout_dataset import extract_layout_v5_5

if FORCE_FROM_SCRATCH['labels_pkl'] and os.path.exists(LABELS_PKL):
    print(f'FORCE_FROM_SCRATCH["labels_pkl"]=True → deleting {LABELS_PKL}')
    os.remove(LABELS_PKL)

if os.path.exists(LABELS_PKL):
    print(f'✓ Labels pkl exists, reusing.')
    print(f'  size: {os.path.getsize(LABELS_PKL)/1e6:.1f} MB')
    with open(LABELS_PKL, 'rb') as f: _labels_peek = pickle.load(f)
    if not isinstance(_labels_peek, list):
        _labels_peek = _labels_peek.get('labels', _labels_peek)
    print(f'  {len(_labels_peek):,} labels')
    del _labels_peek
else:
    df = pd.read_csv(AUGMENTED_CSV)
    print(f'Extracting labels from {len(df):,} rows...')
    labels = []
    rejections = defaultdict(int)
    t0 = time.time()
    rows = df.to_dict('records')
    for idx, row in enumerate(rows):
        label, reason = extract_layout_v5_5(row[SMILES_COL])
        if label is None:
            rejections[reason] += 1
            continue
        label['condition'] = np.array(
            [float(row[c]) for c in cond_cols_resolved], dtype=np.float32)
        labels.append(label)
        if (idx + 1) % 10000 == 0:
            elapsed = time.time() - t0
            eta_m = (len(rows) - idx - 1) / ((idx + 1) / elapsed) / 60
            print(f'  {idx+1:>7,}/{len(rows):,}  kept={len(labels):>7,}  '
                  f'ret={100*len(labels)/(idx+1):5.1f}%  eta={eta_m:.1f}m')
    n_kept = len(labels); n_total = len(rows)
    print(f'\nDone in {(time.time()-t0)/60:.1f} min. Kept {n_kept:,}/{n_total:,} ({100*n_kept/n_total:.2f}%)')
    with open(LABELS_PKL, 'wb') as f: pickle.dump(labels, f)
    print(f'Saved → {LABELS_PKL}  ({os.path.getsize(LABELS_PKL)/1e6:.1f} MB)')

✓ Labels pkl exists, reusing.
  size: 3243.9 MB
  232,934 labels


## 5. Verify environment + label schema

Sanity-checks that PyTorch sees CUDA, the v5.5 vocab constants are correctly imported, and the labels.pkl has all required fields for every stage. **If any required field is missing**, re-run cell 4 with `FORCE_FROM_SCRATCH['labels_pkl']=True`.

In [11]:
import torch, numpy as np, pickle

print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device:       {torch.cuda.get_device_name(0)}')
    print(f'  mem total:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from meanflow.ring_layout_decoder import R_MAX, L_MAX, M_MAX, B_LEN_MAX
from meanflow.ring_layout_diffusion_v5_5 import (
    N_R_CLASSES, N_F_CLASSES, N_L_CLASSES, N_SPIRO_POS_CLASSES,
)
from meanflow.branch_topology_diffusion import P_MAX as A3_P_MAX
from meanflow.ring_atom_diffusion import N_ATOM_CLASSES

print(f'\nv5.5 constants:')
print(f'  R_MAX={R_MAX}, L_MAX={L_MAX}, M_MAX={M_MAX}, B_LEN_MAX={B_LEN_MAX}')
print(f'  A1 vocab: R={N_R_CLASSES}, F={N_F_CLASSES}, L={N_L_CLASSES}, spiro={N_SPIRO_POS_CLASSES}')
print(f'  A3 P_MAX={A3_P_MAX}, A2 atom vocab N_ATOM_CLASSES={N_ATOM_CLASSES}')
print(f'  Terminal num_fragments={TERM_NUM_FRAGMENTS}')

with open(LABELS_PKL, 'rb') as f:
    labels = pickle.load(f)
if not isinstance(labels, list):
    labels = labels.get('labels', labels)
print(f'\nLoaded {len(labels):,} labels')

required = ('R', 'F', 'L', 'B_size', 'B_pos', 'B_parent', 'B_bond',
            'spiro_atom_positions', 'atom_ids', 'M_total', 'terminals', 'condition')
lab0 = labels[0]
missing = [k for k in required if k not in lab0]
if missing:
    print(f'\n⚠️  MISSING fields: {missing}')
    print(f'   Re-run cell 4 with FORCE_FROM_SCRATCH["labels_pkl"]=True')
else:
    print('\n✓ All v5.5 fields present.')

PyTorch:        2.10.0+cu128
CUDA available: True
  device:       NVIDIA RTX PRO 6000 Blackwell Server Edition
  mem total:    102.0 GB

v5.5 constants:
  R_MAX=6, L_MAX=10, M_MAX=40, B_LEN_MAX=15
  A1 vocab: R=11, F=4, L=11, spiro=8
  A3 P_MAX=8, A2 atom vocab N_ATOM_CLASSES=16
  Terminal num_fragments=22

Loaded 232,934 labels

✓ All v5.5 fields present.


---

# Stage 1A — A1 (Ring Layout Diffusion)

**A1's job:** given a property condition, generate the ring-level macro-layout `(R, F, L, spiro_pos)` — what rings exist, which are fused/linked/spiro, what linkers connect them, where spiro atoms sit.

**Architecture:** masked discrete-diffusion transformer over 21 tokens (6 ring + 15 pair). 4 output heads. ~3.6M params at capacity='3M'.

**Target metrics by epoch 25:** `acc_R≥0.85`, `acc_F≥0.95`, `acc_L≥0.90`, `acc_Sp≥0.99`, `decode_rate≥70%`.

## 6. A1 smoke test (1 epoch, capacity=1M)

In [12]:
import shutil

if FORCE_FROM_SCRATCH['a1_smoke_ckpt'] and os.path.isdir(A1_SMOKE_CKPT):
    print(f'FORCE_FROM_SCRATCH["a1_smoke_ckpt"]=True → deleting {A1_SMOKE_CKPT}')
    shutil.rmtree(A1_SMOKE_CKPT)

smoke_best = os.path.join(A1_SMOKE_CKPT, 'best_model.pt')
if os.path.exists(smoke_best):
    print(f'✓ A1 smoke complete. Skipping.')
else:
    from run_training_v5_5_a1 import train_a1
    result = train_a1(
        labels_pkl_path=LABELS_PKL, ckpt_dir=A1_SMOKE_CKPT,
        num_epochs=2, batch_size=128, capacity='1M',
        lr=3e-4, warmup_steps=200, val_fraction=0.05, seed=SEED,
        cfg_drop_prob=0.10, num_workers=0,
        eval_every_n_epochs=1, eval_n_samples=32, eval_n_steps=20,
    )
    print(f'\nA1 smoke complete. best_val_loss = {result["best_val_loss"]:.4f}')

✓ A1 smoke complete. Skipping.


## 7. A1 full training

Wall-clock ~2 hours on L4 at capacity=3M. Auto-resumes from `latest.pt` if interrupted.

In [ ]:
import shutil

if FORCE_FROM_SCRATCH['a1_ckpt']:
    if os.path.isdir(A1_CKPT) and os.listdir(A1_CKPT):
        print(f'⚠️  FORCE_FROM_SCRATCH["a1_ckpt"]=True → DELETING {A1_CKPT}')
        shutil.rmtree(A1_CKPT); os.makedirs(A1_CKPT, exist_ok=True)
else:
    latest = os.path.join(A1_CKPT, 'latest.pt')
    if os.path.exists(latest):
        import torch as _t
        _ck = _t.load(latest, map_location='cpu')
        print(f'✓ A1 will auto-resume from epoch {_ck["epoch"]+1}')
        del _ck
    else:
        print(f'✓ Starting A1 fresh from epoch 0.')

best_path = os.path.join(A1_CKPT, 'best_model.pt')
history_path = os.path.join(A1_CKPT, 'history.json')
if os.path.exists(best_path) and os.path.exists(history_path):
    import json as _j
    with open(history_path) as f: _h = _j.load(f)
    if _h and _h[-1]['epoch'] >= A1_NUM_EPOCHS - 1:
        print(f'✓ A1 already trained for {A1_NUM_EPOCHS} epochs.')
        print(f'  Set FORCE_FROM_SCRATCH["a1_ckpt"]=True to retrain.')
    else:
        from run_training_v5_5_a1 import train_a1
        result = train_a1(
            labels_pkl_path=LABELS_PKL, ckpt_dir=A1_CKPT,
            num_epochs=A1_NUM_EPOCHS, batch_size=BATCH_SIZE,
            capacity=A1_CAPACITY, lr=LR, warmup_steps=200,
            val_fraction=VAL_FRACTION, seed=SEED,
            cfg_drop_prob=A1_CFG_DROP_PROB, num_workers=2,
            eval_every_n_epochs=1, eval_n_samples=64, eval_n_steps=20,
            grad_clip=1.0, ema_decay=0.999, weight_decay=0.0,
        )
        print(f'\nA1 done. best_val_loss = {result["best_val_loss"]:.4f}')
else:
    from run_training_v5_5_a1 import train_a1
    result = train_a1(
        labels_pkl_path=LABELS_PKL, ckpt_dir=A1_CKPT,
        num_epochs=A1_NUM_EPOCHS, batch_size=BATCH_SIZE,
        capacity=A1_CAPACITY, lr=LR, warmup_steps=200,
        val_fraction=VAL_FRACTION, seed=SEED,
        cfg_drop_prob=A1_CFG_DROP_PROB, num_workers=2,
        eval_every_n_epochs=1, eval_n_samples=64, eval_n_steps=20,
        grad_clip=1.0, ema_decay=0.999, weight_decay=0.0,
    )
    print(f'\nA1 done. best_val_loss = {result["best_val_loss"]:.4f}')

## 8. A1 training-history plot

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

with open(os.path.join(A1_CKPT, 'history.json')) as f: h = json.load(f)
epochs = [r['epoch'] for r in h]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, [r['train_loss'] for r in h], label='train')
axes[0].plot(epochs, [r['val_loss']   for r in h], label='val', marker='o', markersize=3)
axes[0].set_title('A1: loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

for k in ('acc_R', 'acc_F', 'acc_L', 'acc_Spiro'):
    axes[1].plot(epochs, [r['train_metrics'].get(k, float('nan')) for r in h], label=k)
axes[1].set_title('A1: per-head accuracy'); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.05)

axes[2].plot(epochs, [r.get('decode_rate', float('nan')) for r in h],
             marker='o', markersize=3)
axes[2].set_title('A1: decode_rate'); axes[2].grid(alpha=0.3); axes[2].set_ylim(0, 1.05)

plt.tight_layout(); plt.show()
best_idx = int(np.argmin([r['val_loss'] for r in h]))
print(f'A1 best: val_loss={h[best_idx]["val_loss"]:.4f} at epoch {h[best_idx]["epoch"]}, '
      f'decode_rate={h[best_idx].get("decode_rate", 0)*100:.1f}%')

---

# Stage 1B — A3 (Branch Topology)

**A3's job:** given A1's ring layout + condition, predict the branch trees `(B_size, B_pos, B_parent, B_bond)` hanging off each ring. This is what turns a ring-level scaffold into an atom-level scaffold.

**Architecture:** one-pass classifier over 69 tokens (6 ring + 15 pair + 48 slot). Different from A1/A2/Terminal — no diffusion, single forward pass + post-processing. ~19M params at capacity='10M'.

**Target metrics by epoch 25:** `acc_size≥0.98`, `acc_pos≥0.70`, `acc_parent≥0.95`, `acc_bond≥0.97`. The position accuracy is the slowest because ring positions are often chemically symmetric (e.g. positions 2/6 on a benzene are equivalent), so the model gets 'wrong' on symmetric positions while producing identical molecules. Don't worry if `acc_pos` ceilings at 0.70.

**Note:** A3 is the longest stage to train — ~10 hours on L4. Plan to run this across multiple Colab sessions; **auto-resume handles disconnects**.

## 9. A3 smoke test (1 epoch, capacity=3M)

In [15]:
import shutil

if FORCE_FROM_SCRATCH['a3_smoke_ckpt'] and os.path.isdir(A3_SMOKE_CKPT):
    print(f'FORCE_FROM_SCRATCH["a3_smoke_ckpt"]=True → deleting {A3_SMOKE_CKPT}')
    shutil.rmtree(A3_SMOKE_CKPT)

smoke_best = os.path.join(A3_SMOKE_CKPT, 'best_model.pt')
if os.path.exists(smoke_best):
    print(f'✓ A3 smoke complete. Skipping.')
else:
    from run_training_v5_5_a3 import train_a3
    result = train_a3(
        labels_pkl_path=LABELS_PKL, ckpt_dir=A3_SMOKE_CKPT,
        num_epochs=1, batch_size=64, capacity='3M',
        num_workers=0,
        val_every_n_epochs=1, checkpoint_every_n_epochs=1,
        n_val_batches=2, seed=SEED,
    )
    print(f'\nA3 smoke complete. best_val_loss = {result["best_val_loss"]:.4f}')

✓ A3 smoke complete. Skipping.


## 10. A3 full training (~10 hours; spans multiple sessions)

In [ ]:
import shutil

if FORCE_FROM_SCRATCH['a3_ckpt']:
    if os.path.isdir(A3_CKPT) and os.listdir(A3_CKPT):
        print(f'⚠️  FORCE_FROM_SCRATCH["a3_ckpt"]=True → DELETING {A3_CKPT}')
        shutil.rmtree(A3_CKPT); os.makedirs(A3_CKPT, exist_ok=True)
else:
    latest = os.path.join(A3_CKPT, 'latest.pt')
    if os.path.exists(latest):
        import torch as _t
        _ck = _t.load(latest, map_location='cpu')
        print(f'✓ A3 will auto-resume from epoch {_ck["epoch"]+1}')
        del _ck
    else:
        print(f'✓ Starting A3 fresh from epoch 0.')

best_path = os.path.join(A3_CKPT, 'best_model.pt')
history_path = os.path.join(A3_CKPT, 'history.json')
if os.path.exists(best_path) and os.path.exists(history_path):
    import json as _j
    with open(history_path) as f: _h = _j.load(f)
    if _h and _h[-1]['epoch'] >= A3_NUM_EPOCHS - 1:
        print(f'✓ A3 already trained for {A3_NUM_EPOCHS} epochs.')
        print(f'  Set FORCE_FROM_SCRATCH["a3_ckpt"]=True to retrain.')
    else:
        from run_training_v5_5_a3 import train_a3
        result = train_a3(
            labels_pkl_path=LABELS_PKL, ckpt_dir=A3_CKPT,
            num_epochs=A3_NUM_EPOCHS, batch_size=BATCH_SIZE,
            learning_rate=LR, weight_decay=0.01,
            warmup_frac=0.05, min_lr_frac=0.1,
            capacity=A3_CAPACITY, val_frac=VAL_FRACTION,
            cfg_drop_prob=A3_CFG_DROP_PROB, grad_clip=1.0,
            ema_decay=0.999, seed=SEED,
            val_every_n_epochs=1, checkpoint_every_n_epochs=5,
            n_val_batches=4, num_workers=2,
        )
        print(f'\nA3 done. best_val_loss = {result["best_val_loss"]:.4f}')
else:
    from run_training_v5_5_a3 import train_a3
    result = train_a3(
        labels_pkl_path=LABELS_PKL, ckpt_dir=A3_CKPT,
        num_epochs=A3_NUM_EPOCHS, batch_size=BATCH_SIZE,
        learning_rate=LR, weight_decay=0.01,
        warmup_frac=0.05, min_lr_frac=0.1,
        capacity=A3_CAPACITY, val_frac=VAL_FRACTION,
        cfg_drop_prob=A3_CFG_DROP_PROB, grad_clip=1.0,
        ema_decay=0.999, seed=SEED,
        val_every_n_epochs=1, checkpoint_every_n_epochs=5,
        n_val_batches=4, num_workers=2,
    )
    print(f'\nA3 done. best_val_loss = {result["best_val_loss"]:.4f}')

## 11. A3 training-history plot

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

with open(os.path.join(A3_CKPT, 'history.json')) as f: h = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(h['epoch'], h['train_loss'], label='train')
val_eps = [e for e, v in zip(h['epoch'], h['val_loss']) if isinstance(v, float) and v == v]
val_loss = [v for v in h['val_loss'] if isinstance(v, float) and v == v]
if val_loss: axes[0].plot(val_eps, val_loss, label='val', marker='o', markersize=3)
axes[0].set_title('A3: loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

for k in ('acc_size', 'acc_pos', 'acc_parent', 'acc_bond'):
    if k in h: axes[1].plot(h['epoch'], h[k], label=k)
axes[1].set_title('A3: per-head accuracy'); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.05)

val_metrics = h.get('val_metrics', [])
filled = [(e, m) for e, m in zip(h['epoch'], val_metrics) if m]
if filled:
    eps, mets = zip(*filled)
    for k in ('rate_pos_in_ring', 'rate_parent_causal', 'rate_bond_nonzero'):
        ys = [m.get(k, float('nan')) for m in mets]
        axes[2].plot(eps, ys, label=k, marker='o', markersize=3)
axes[2].set_title('A3: validity rates'); axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3); axes[2].set_ylim(0, 1.05)

plt.tight_layout(); plt.show()
if h.get('val_loss'):
    losses = [v for v in h['val_loss'] if isinstance(v, float) and v == v]
    if losses:
        bi = int(np.argmin(losses))
        print(f'A3 best: val_loss={losses[bi]:.4f} at epoch {val_eps[bi]}')

---

# Stage 1C — A2 (Atom Assignment)

**A2's job:** given the decoded scaffold's bond skeleton (from A1+A3 outputs) and the condition, predict the element identity at each atom position.

**Architecture:** masked discrete-diffusion transformer with edge-biased attention. The attention mechanism is conditioned on the scaffold bond matrix via a learned `(n_heads, n_bond_classes)` bias table. ~8.5M params at capacity='10M'.

**Vocabulary:** K=16 atom classes including Phase 2D charged classes (`O-`, `N+`, `n+`, `N-`, `n-`, `P+`). Bond classes K=5 (`none/single/aromatic/double/triple`) to match the v5.5 encoder.

**Target metrics by epoch 25:** `acc≥0.92`, `sample_acc≥0.80`, `arom≥0.95` (aromatic-constraint compliance), `aliph≥0.90`.

## 12. A2 smoke test

In [18]:
import shutil

if FORCE_FROM_SCRATCH['a2_smoke_ckpt'] and os.path.isdir(A2_SMOKE_CKPT):
    print(f'FORCE_FROM_SCRATCH["a2_smoke_ckpt"]=True → deleting {A2_SMOKE_CKPT}')
    shutil.rmtree(A2_SMOKE_CKPT)

smoke_best = os.path.join(A2_SMOKE_CKPT, 'best_model.pt')
if os.path.exists(smoke_best):
    print(f'✓ A2 smoke complete. Skipping.')
else:
    from run_training_v5_5_a2 import train_a2
    result = train_a2(
        labels_pkl_path=LABELS_PKL, ckpt_dir=A2_SMOKE_CKPT,
        num_epochs=2, batch_size=64, capacity='1M',
        lr=3e-4, warmup_steps=200, val_fraction=0.05, seed=SEED,
        cfg_drop_prob=0.10, num_workers=0,
        eval_every_n_epochs=1, eval_n_samples=32, eval_n_steps=20,
    )
    print(f'\nA2 smoke complete. best_val_loss = {result["best_val_loss"]:.4f}')

✓ A2 smoke complete. Skipping.


## 13. A2 full training

In [ ]:
import shutil

if FORCE_FROM_SCRATCH['a2_ckpt']:
    if os.path.isdir(A2_CKPT) and os.listdir(A2_CKPT):
        print(f'⚠️  FORCE_FROM_SCRATCH["a2_ckpt"]=True → DELETING {A2_CKPT}')
        shutil.rmtree(A2_CKPT); os.makedirs(A2_CKPT, exist_ok=True)
else:
    latest = os.path.join(A2_CKPT, 'latest.pt')
    if os.path.exists(latest):
        import torch as _t
        _ck = _t.load(latest, map_location='cpu')
        print(f'✓ A2 will auto-resume from epoch {_ck["epoch"]+1}')
        del _ck
    else:
        print(f'✓ Starting A2 fresh from epoch 0.')

best_path = os.path.join(A2_CKPT, 'best_model.pt')
history_path = os.path.join(A2_CKPT, 'history.json')
if os.path.exists(best_path) and os.path.exists(history_path):
    import json as _j
    with open(history_path) as f: _h = _j.load(f)
    if _h and _h[-1]['epoch'] >= A2_NUM_EPOCHS - 1:
        print(f'✓ A2 already trained for {A2_NUM_EPOCHS} epochs.')
        print(f'  Set FORCE_FROM_SCRATCH["a2_ckpt"]=True to retrain.')
    else:
        from run_training_v5_5_a2 import train_a2
        result = train_a2(
            labels_pkl_path=LABELS_PKL, ckpt_dir=A2_CKPT,
            num_epochs=A2_NUM_EPOCHS, batch_size=BATCH_SIZE,
            capacity=A2_CAPACITY, lr=LR, warmup_steps=500,
            val_fraction=VAL_FRACTION, seed=SEED,
            cfg_drop_prob=A2_CFG_DROP_PROB, num_workers=2,
            eval_every_n_epochs=1, eval_n_samples=64, eval_n_steps=20,
            grad_clip=1.0, ema_decay=0.999, weight_decay=0.0,
        )
        print(f'\nA2 done. best_val_loss = {result["best_val_loss"]:.4f}')
else:
    from run_training_v5_5_a2 import train_a2
    result = train_a2(
        labels_pkl_path=LABELS_PKL, ckpt_dir=A2_CKPT,
        num_epochs=A2_NUM_EPOCHS, batch_size=BATCH_SIZE,
        capacity=A2_CAPACITY, lr=LR, warmup_steps=500,
        val_fraction=VAL_FRACTION, seed=SEED,
        cfg_drop_prob=A2_CFG_DROP_PROB, num_workers=2,
        eval_every_n_epochs=1, eval_n_samples=64, eval_n_steps=20,
        grad_clip=1.0, ema_decay=0.999, weight_decay=0.0,
    )
    print(f'\nA2 done. best_val_loss = {result["best_val_loss"]:.4f}')

## 14. A2 training-history plot

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

with open(os.path.join(A2_CKPT, 'history.json')) as f: h = json.load(f)
epochs = [r['epoch'] for r in h]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, [r['train_loss'] for r in h], label='train')
axes[0].plot(epochs, [r['val_loss']   for r in h], label='val', marker='o', markersize=3)
axes[0].set_title('A2: loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [r['train_metrics'].get('acc', float('nan')) for r in h], label='train')
axes[1].plot(epochs, [r['val_metrics'].get('acc', float('nan'))   for r in h], label='val', marker='o', markersize=3)
axes[1].set_title('A2: atom-id accuracy'); axes[1].legend()
axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.05)

for k in ('sample_atom_acc', 'sample_arom_compliance', 'sample_aliph_pick_rate'):
    ys = [r.get(k, float('nan')) for r in h]
    axes[2].plot(epochs, ys, label=k, marker='o', markersize=3)
axes[2].set_title('A2: sample-quality eval'); axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3); axes[2].set_ylim(0, 1.05)

plt.tight_layout(); plt.show()
best_idx = int(np.argmin([r['val_loss'] for r in h]))
print(f'A2 best: val_loss={h[best_idx]["val_loss"]:.4f} at epoch {h[best_idx]["epoch"]}')

---

# Stage 2 — Terminal (Fragment Decoration)

**Terminal's job:** given the fully-assembled scaffold (atom_ids + bond_classes + atom_mask from Stage 1) and the condition, predict a fragment id (or class 0 = no decoration) per scaffold atom.

**Architecture:** masked discrete-diffusion transformer with bond-biased attention (same family as A2). ~8.6M params at capacity='9M'.

**Vocabulary:** K=22 terminal classes (v5.5-ZINC expansion). v5.4 RedDB had K=9; the 13 new classes are Cl, Br, I, CN, NO2, OCH3, CF3, Thiol, AcylHalide, Cyanate, Thiocyanate, Isothiocyanate, Isonitrile. Atom vocab K=16 (Phase 2D charged classes).

**Target metrics by epoch 25:** `acc≥0.95` (dominated by class-0 'no decoration'), `nz_acc≥0.50` (the real signal, on positions with target ≠ 0). The class imbalance is extreme: top-2 classes (CH3, =O) account for 73% of all decorated atoms in ZINC. Inverse-frequency class weights help but cause some over-prediction of rare classes.

## 15. Terminal smoke test

In [21]:
import shutil

if FORCE_FROM_SCRATCH['term_smoke_ckpt'] and os.path.isdir(TERM_SMOKE_CKPT):
    print(f'FORCE_FROM_SCRATCH["term_smoke_ckpt"]=True → deleting {TERM_SMOKE_CKPT}')
    shutil.rmtree(TERM_SMOKE_CKPT)

smoke_best = os.path.join(TERM_SMOKE_CKPT, 'best_model.pt')
if os.path.exists(smoke_best):
    print(f'✓ Terminal smoke complete. Skipping.')
else:
    from run_training_v5_5_terminal import train_terminal
    result = train_terminal(
        labels_pkl_path=LABELS_PKL, ckpt_dir=TERM_SMOKE_CKPT,
        num_epochs=2, batch_size=64, capacity='1M',
        num_fragments=TERM_NUM_FRAGMENTS,
        lr=3e-4, val_fraction=0.05, seed=SEED, num_workers=0,
        use_class_weights=TERM_USE_CLASS_WEIGHTS,
        eval_every_n_epochs=1, eval_n_samples=32,
    )
    print(f'\nTerminal smoke complete. best_val_loss = {result["best_val_loss"]:.4f}')

✓ Terminal smoke complete. Skipping.


## 16. Terminal full training

In [ ]:
import shutil

if FORCE_FROM_SCRATCH['term_ckpt']:
    if os.path.isdir(TERM_CKPT) and os.listdir(TERM_CKPT):
        print(f'⚠️  FORCE_FROM_SCRATCH["term_ckpt"]=True → DELETING {TERM_CKPT}')
        shutil.rmtree(TERM_CKPT); os.makedirs(TERM_CKPT, exist_ok=True)
else:
    latest = os.path.join(TERM_CKPT, 'latest.pt')
    if os.path.exists(latest):
        import torch as _t
        _ck = _t.load(latest, map_location='cpu')
        print(f'✓ Terminal will auto-resume from epoch {_ck["epoch"]+1}')
        del _ck
    else:
        print(f'✓ Starting Terminal fresh from epoch 0.')

best_path = os.path.join(TERM_CKPT, 'best_model.pt')
history_path = os.path.join(TERM_CKPT, 'history.json')
if os.path.exists(best_path) and os.path.exists(history_path):
    import json as _j
    with open(history_path) as f: _h = _j.load(f)
    if _h and _h[-1]['epoch'] >= TERM_NUM_EPOCHS - 1:
        print(f'✓ Terminal already trained for {TERM_NUM_EPOCHS} epochs.')
        print(f'  Set FORCE_FROM_SCRATCH["term_ckpt"]=True to retrain.')
    else:
        from run_training_v5_5_terminal import train_terminal
        result = train_terminal(
            labels_pkl_path=LABELS_PKL, ckpt_dir=TERM_CKPT,
            num_epochs=TERM_NUM_EPOCHS, batch_size=BATCH_SIZE,
            capacity=TERM_CAPACITY,
            num_fragments=TERM_NUM_FRAGMENTS,
            lr=LR, warmup_steps=500, val_fraction=VAL_FRACTION,
            seed=SEED, use_class_weights=TERM_USE_CLASS_WEIGHTS,
            num_workers=2,
            eval_every_n_epochs=1, eval_n_samples=64,
            grad_clip=1.0, ema_decay=0.999, weight_decay=0.0,
        )
        print(f'\nTerminal done. best_val_loss = {result["best_val_loss"]:.4f}')
else:
    from run_training_v5_5_terminal import train_terminal
    result = train_terminal(
        labels_pkl_path=LABELS_PKL, ckpt_dir=TERM_CKPT,
        num_epochs=TERM_NUM_EPOCHS, batch_size=BATCH_SIZE,
        capacity=TERM_CAPACITY,
        num_fragments=TERM_NUM_FRAGMENTS,
        lr=LR, warmup_steps=500, val_fraction=VAL_FRACTION,
        seed=SEED, use_class_weights=TERM_USE_CLASS_WEIGHTS,
        num_workers=2,
        eval_every_n_epochs=1, eval_n_samples=64,
        grad_clip=1.0, ema_decay=0.999, weight_decay=0.0,
    )
    print(f'\nTerminal done. best_val_loss = {result["best_val_loss"]:.4f}')

## 17. Terminal training-history plot

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

with open(os.path.join(TERM_CKPT, 'history.json')) as f: h = json.load(f)
epochs = [r['epoch'] for r in h]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, [r['train_loss'] for r in h], label='train')
axes[0].plot(epochs, [r['val_loss']   for r in h], label='val', marker='o', markersize=3)
axes[0].set_title('Terminal: loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

for k in ('overall_acc', 'nonzero_acc'):
    ys = [r['train_metrics'].get(k, float('nan')) for r in h]
    axes[1].plot(epochs, ys, label=f'train {k}')
    ys = [r['val_metrics'].get(k, float('nan')) for r in h]
    axes[1].plot(epochs, ys, label=f'val {k}', marker='o', markersize=3)
axes[1].set_title('Terminal: accuracies'); axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.05)

for k in ('atom_acc', 'nonzero_acc', 'precision'):
    ys = [r.get('eval_metrics', {}).get(k, float('nan')) for r in h]
    axes[2].plot(epochs, ys, label=k, marker='o', markersize=3)
axes[2].set_title('Terminal: sample-quality eval'); axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3); axes[2].set_ylim(0, 1.05)

plt.tight_layout(); plt.show()
best_idx = int(np.argmin([r['val_loss'] for r in h]))
print(f'Terminal best: val_loss={h[best_idx]["val_loss"]:.4f} at epoch {h[best_idx]["epoch"]}')

---

## Training complete

All four `best_model.pt` checkpoints should now exist:

- `checkpoints_a1_v5_5_3M/best_model.pt`
- `checkpoints_a3_v5_5_10M/best_model.pt`
- `checkpoints_a2_v5_5_10M/best_model.pt`
- `checkpoints_terminal_v5_5_9M/best_model.pt`

**Next:** open `02_evaluation.ipynb` to run end-to-end generation + V·U·N evaluation.

If you want to retrain a stage with different hyperparameters: set its `FORCE_FROM_SCRATCH` flag in cell 1 and re-run that stage's cells.